In [ ]:
{
 "cells": [
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "# 03. Model Training\n",
    "## NLP Chatbot Project - Training 4 Models\n",
    "\n",
    "This notebook trains and compares 4 machine learning models:\n",
    "1. Logistic Regression\n",
    "2. Random Forest\n",
    "3. XGBoost\n",
    "4. LSTM Neural Network\n",
    "\n",
    "**Goal**: Train all models and select the best performer for deployment"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Import libraries\n",
    "import pandas as pd\n",
    "import numpy as np\n",
    "import matplotlib.pyplot as plt\n",
    "import seaborn as sns\n",
    "import sys\n",
    "import time\n",
    "import pickle\n",
    "import warnings\n",
    "warnings.filterwarnings('ignore')\n",
    "\n",
    "# Add src to path\n",
    "sys.path.append('../src')\n",
    "\n",
    "# Import custom modules\n",
    "from preprocessor import TextVectorizer\n",
    "from models import (\n",
    "    LogisticRegressionModel,\n",
    "    RandomForestModel,\n",
    "    XGBoostModel,\n",
    "    LSTMModel\n",
    ")\n",
    "from sklearn.preprocessing import LabelEncoder\n",
    "\n",
    "# Set style\n",
    "sns.set_style('whitegrid')\n",
    "plt.rcParams['figure.figsize'] = (12, 6)\n",
    "\n",
    "print(\"✅ Imports successful!\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 1. Load Preprocessed Data"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Load train, validation, and test data\n",
    "print(\"📂 Loading preprocessed data...\\n\")\n",
    "\n",
    "train_df = pd.read_csv('../data/train.csv')\n",
    "val_df = pd.read_csv('../data/val.csv')\n",
    "test_df = pd.read_csv('../data/test.csv')\n",
    "\n",
    "print(f\"✅ Data loaded successfully!\")\n",
    "print(f\"   Training:   {len(train_df):,} samples\")\n",
    "print(f\"   Validation: {len(val_df):,} samples\")\n",
    "print(f\"   Test:       {len(test_df):,} samples\")\n",
    "print(f\"\\nColumns: {train_df.columns.tolist()}\")\n",
    "print(f\"\\nSample data:\")\n",
    "train_df.head()"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 2. Prepare Labels"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Encode labels\n",
    "print(\"🏷️ Encoding labels...\\n\")\n",
    "\n",
    "label_encoder = LabelEncoder()\n",
    "\n",
    "# Check if labels are already encoded\n",
    "if 'label_encoded' not in train_df.columns:\n",
    "    y_train = label_encoder.fit_transform(train_df['label'])\n",
    "    y_val = label_encoder.transform(val_df['label'])\n",
    "    y_test = label_encoder.transform(test_df['label'])\n",
    "else:\n",
    "    y_train = train_df['label_encoded'].values\n",
    "    y_val = val_df['label_encoded'].values\n",
    "    y_test = test_df['label_encoded'].values\n",
    "    # Fit label encoder for later use\n",
    "    label_encoder.fit(train_df['label'])\n",
    "\n",
    "num_classes = len(label_encoder.classes_)\n",
    "\n",
    "print(f\"✅ Labels encoded!\")\n",
    "print(f\"   Number of classes: {num_classes}\")\n",
    "print(f\"   Label distribution:\")\n",
    "print(f\"   - Training: {np.bincount(y_train)}\")\n",
    "print(f\"   - Validation: {np.bincount(y_val)}\")\n",
    "print(f\"   - Test: {np.bincount(y_test)}\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 3. Vectorize Text (for Traditional Models)"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Initialize TF-IDF vectorizer\n",
    "print(\"📊 Vectorizing text with TF-IDF...\\n\")\n",
    "\n",
    "vectorizer = TextVectorizer(\n",
    "    method='tfidf',\n",
    "    max_features=5000,\n",
    "    ngram_range=(1, 2)\n",
    ")\n",
    "\n",
    "# Fit and transform\n",
    "X_train = vectorizer.fit_transform(train_df['processed_text'])\n",
    "X_val = vectorizer.transform(val_df['processed_text'])\n",
    "X_test = vectorizer.transform(test_df['processed_text'])\n",
    "\n",
    "print(f\"✅ Vectorization complete!\")\n",
    "print(f\"   Training shape:   {X_train.shape}\")\n",
    "print(f\"   Validation shape: {X_val.shape}\")\n",
    "print(f\"   Test shape:       {X_test.shape}\")\n",
    "print(f\"   Sparsity: {(1 - X_train.nnz / (X_train.shape[0] * X_train.shape[1]))*100:.2f}%\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 4. MODEL 1: Logistic Regression"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "print(\"=\"*80)\n",
    "print(\"MODEL 1/4: LOGISTIC REGRESSION\")\n",
    "print(\"=\"*80)\n",
    "\n",
    "# Initialize model\n",
    "lr_model = LogisticRegressionModel(max_iter=1000, C=1.0)\n",
    "\n",
    "# Train\n",
    "start_time = time.time()\n",
    "lr_model.train(X_train, y_train)\n",
    "train_time = time.time() - start_time\n",
    "\n",
    "# Evaluate on validation set\n",
    "val_predictions = lr_model.predict(X_val)\n",
    "val_accuracy = (val_predictions == y_val).mean()\n",
    "\n",
    "print(f\"\\n✅ Training complete!\")\n",
    "print(f\"   Training time: {train_time:.2f} seconds\")\n",
    "print(f\"   Validation accuracy: {val_accuracy:.4f}\")\n",
    "\n",
    "# Save model\n",
    "lr_model.save('../models/logistic_model.pkl')\n",
    "print(f\"   Model saved to: ../models/logistic_model.pkl\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 5. MODEL 2: Random Forest"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "print(\"\\n\" + \"=\"*80)\n",
    "print(\"MODEL 2/4: RANDOM FOREST\")\n",
    "print(\"=\"*80)\n",
    "\n",
    "# Initialize model\n",
    "rf_model = RandomForestModel(n_estimators=200, max_depth=30)\n",
    "\n",
    "# Train\n",
    "start_time = time.time()\n",
    "rf_model.train(X_train, y_train)\n",
    "train_time = time.time() - start_time\n",
    "\n",
    "# Evaluate on validation set\n",
    "val_predictions = rf_model.predict(X_val)\n",
    "val_accuracy = (val_predictions == y_val).mean()\n",
    "\n",
    "print(f\"\\n✅ Training complete!\")\n",
    "print(f\"   Training time: {train_time:.2f} seconds\")\n",
    "print(f\"   Validation accuracy: {val_accuracy:.4f}\")\n",
    "\n",
    "# Save model\n",
    "rf_model.save('../models/random_forest_model.pkl')\n",
    "print(f\"   Model saved to: ../models/random_forest_model.pkl\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 6. MODEL 3: XGBoost"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "print(\"\\n\" + \"=\"*80)\n",
    "print(\"MODEL 3/4: XGBOOST\")\n",
    "print(\"=\"*80)\n",
    "\n",
    "# Initialize model\n",
    "xgb_model = XGBoostModel(n_estimators=200, max_depth=10, learning_rate=0.1)\n",
    "\n",
    "# Train with validation monitoring\n",
    "start_time = time.time()\n",
    "xgb_model.train(X_train, y_train, X_val, y_val)\n",
    "train_time = time.time() - start_time\n",
    "\n",
    "# Evaluate on validation set\n",
    "val_predictions = xgb_model.predict(X_val)\n",
    "val_accuracy = (val_predictions == y_val).mean()\n",
    "\n",
    "print(f\"\\n✅ Training complete!\")\n",
    "print(f\"   Training time: {train_time:.2f} seconds\")\n",
    "print(f\"   Validation accuracy: {val_accuracy:.4f}\")\n",
    "\n",
    "# Save model\n",
    "xgb_model.save('../models/xgboost_model.pkl')\n",
    "print(f\"   Model saved to: ../models/xgboost_model.pkl\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 7. MODEL 4: LSTM Neural Network"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "print(\"\\n\" + \"=\"*80)\n",
    "print(\"MODEL 4/4: LSTM NEURAL NETWORK\")\n",
    "print(\"=\"*80)\n",
    "\n",
    "# Initialize model\n",
    "lstm_model = LSTMModel(\n",
    "    vocab_size=10000,\n",
    "    embedding_dim=128,\n",
    "    lstm_units=128,\n",
    "    num_classes=num_classes,\n",
    "    max_length=100\n",
    ")\n",
    "\n",
    "print(\"\\nModel Architecture:\")\n",
    "lstm_model.model.summary()\n",
    "\n",
    "# Prepare text data\n",
    "X_train_texts = train_df['processed_text'].tolist()\n",
    "X_val_texts = val_df['processed_text'].tolist()\n",
    "\n",
    "# Train\n",
    "start_time = time.time()\n",
    "history = lstm_model.train(\n",
    "    X_train_texts, y_train,\n",
    "    X_val_texts, y_val,\n",
    "    epochs=20,\n",
    "    batch_size=32\n",
    ")\n",
    "train_time = time.time() - start_time\n",
    "\n",
    "print(f\"\\n✅ Training complete!\")\n",
    "print(f\"   Training time: {train_time:.2f} seconds\")\n",
    "\n",
    "# Save model\n",
    "lstm_model.save('../models/lstm_model.h5', '../models/lstm_tokenizer.pkl')\n",
    "print(f\"   Model saved to: ../models/lstm_model.h5\")\n",
    "print(f\"   Tokenizer saved to: ../models/lstm_tokenizer.pkl\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Plot LSTM training history\n",
    "fig, axes = plt.subplots(1, 2, figsize=(15, 5))\n",
    "\n",
    "# Accuracy\n",
    "axes[0].plot(history.history['accuracy'], label='Training Accuracy', linewidth=2)\n",
    "axes[0].plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)\n",
    "axes[0].set_title('LSTM Model Accuracy', fontsize=14, fontweight='bold')\n",
    "axes[0].set_xlabel('Epoch')\n",
    "axes[0].set_ylabel('Accuracy')\n",
    "axes[0].legend()\n",
    "axes[0].grid(alpha=0.3)\n",
    "\n",
    "# Loss\n",
    "axes[1].plot(history.history['loss'], label='Training Loss', linewidth=2)\n",
    "axes[1].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)\n",
    "axes[1].set_title('LSTM Model Loss', fontsize=14, fontweight='bold')\n",
    "axes[1].set_xlabel('Epoch')\n",
    "axes[1].set_ylabel('Loss')\n",
    "axes[1].legend()\n",
    "axes[1].grid(alpha=0.3)\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.savefig('../models/lstm_training_history.png', dpi=300, bbox_inches='tight')\n",
    "plt.show()\n",
    "\n",
    "print(\"\\n✅ Training history plot saved\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 8. Quick Validation Comparison"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "from sklearn.metrics import accuracy_score, f1_score\n",
    "\n",
    "print(\"=\"*80)\n",
    "print(\"VALIDATION SET PERFORMANCE\")\n",
    "print(\"=\"*80)\n",
    "\n",
    "# Logistic Regression\n",
    "lr_pred = lr_model.predict(X_val)\n",
    "lr_acc = accuracy_score(y_val, lr_pred)\n",
    "lr_f1 = f1_score(y_val, lr_pred, average='weighted')\n",
    "\n",
    "# Random Forest\n",
    "rf_pred = rf_model.predict(X_val)\n",
    "rf_acc = accuracy_score(y_val, rf_pred)\n",
    "rf_f1 = f1_score(y_val, rf_pred, average='weighted')\n",
    "\n",
    "# XGBoost\n",
    "xgb_pred = xgb_model.predict(X_val)\n",
    "xgb_acc = accuracy_score(y_val, xgb_pred)\n",
    "xgb_f1 = f1_score(y_val, xgb_pred, average='weighted')\n",
    "\n",
    "# LSTM\n",
    "lstm_pred = lstm_model.predict(X_val_texts)\n",
    "lstm_acc = accuracy_score(y_val, lstm_pred)\n",
    "lstm_f1 = f1_score(y_val, lstm_pred, average='weighted')\n",
    "\n",
    "# Create comparison DataFrame\n",
    "results = pd.DataFrame({\n",
    "    'Model': ['Logistic Regression', 'Random Forest', 'XGBoost', 'LSTM'],\n",
    "    'Accuracy': [lr_acc, rf_acc, xgb_acc, lstm_acc],\n",
    "    'F1-Score': [lr_f1, rf_f1, xgb_f1, lstm_f1]\n",
    "})\n",
    "\n",
    "print(\"\\n\", results.to_string(index=False))\n",
    "print(\"\\n\" + \"=\"*80)\n",
    "\n",
    "# Find best model\n",
    "best_model_idx = results['F1-Score'].idxmax()\n",
    "best_model_name = results.loc[best_model_idx, 'Model']\n",
    "best_f1 = results.loc[best_model_idx, 'F1-Score']\n",
    "\n",
    "print(f\"\\n🏆 BEST MODEL (by F1-Score): {best_model_name}\")\n",
    "print(f\"   F1-Score: {best_f1:.4f}\")"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Visualize comparison\n",
    "fig, ax = plt.subplots(figsize=(12, 6))\n",
    "\n",
    "x = np.arange(len(results))\n",
    "width = 0.35\n",
    "\n",
    "bars1 = ax.bar(x - width/2, results['Accuracy'], width, label='Accuracy', color='steelblue')\n",
    "bars2 = ax.bar(x + width/2, results['F1-Score'], width, label='F1-Score', color='orange')\n",
    "\n",
    "ax.set_xlabel('Model', fontsize=12, fontweight='bold')\n",
    "ax.set_ylabel('Score', fontsize=12, fontweight='bold')\n",
    "ax.set_title('Model Performance Comparison (Validation Set)', fontsize=14, fontweight='bold')\n",
    "ax.set_xticks(x)\n",
    "ax.set_xticklabels(results['Model'], rotation=15, ha='right')\n",
    "ax.legend()\n",
    "ax.grid(alpha=0.3, axis='y')\n",
    "ax.set_ylim([0, 1.0])\n",
    "\n",
    "# Add value labels on bars\n",
    "for bars in [bars1, bars2]:\n",
    "    for bar in bars:\n",
    "        height = bar.get_height()\n",
    "        ax.text(bar.get_x() + bar.get_width()/2., height,\n",
    "                f'{height:.3f}',\n",
    "                ha='center', va='bottom', fontsize=9)\n",
    "\n",
    "plt.tight_layout()\n",
    "plt.savefig('../models/model_comparison_validation.png', dpi=300, bbox_inches='tight')\n",
    "plt.show()\n",
    "\n",
    "print(\"\\n✅ Comparison plot saved\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 9. Save Artifacts"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "# Save vectorizer\n",
    "print(\"💾 Saving artifacts...\\n\")\n",
    "\n",
    "vectorizer.save('../models/vectorizer.pkl')\n",
    "print(\"✅ Vectorizer saved to: ../models/vectorizer.pkl\")\n",
    "\n",
    "# Save label encoder\n",
    "with open('../models/label_encoder.pkl', 'wb') as f:\n",
    "    pickle.dump(label_encoder, f)\n",
    "print(\"✅ Label encoder saved to: ../models/label_encoder.pkl\")\n",
    "\n",
    "# Save validation results\n",
    "results.to_csv('../models/validation_results.csv', index=False)\n",
    "print(\"✅ Validation results saved to: ../models/validation_results.csv\")"
   ]
  },
  {
   "cell_type": "markdown",
   "metadata": {},
   "source": [
    "## 10. Summary"
   ]
  },
  {
   "cell_type": "code",
   "execution_count": null,
   "metadata": {},
   "outputs": [],
   "source": [
    "print(\"=\"*80)\n",
    "print(\"TRAINING SUMMARY\")\n",
    "print(\"=\"*80)\n",
    "\n",
    "summary = f\"\"\"\n",
    "✅ All 4 Models Trained Successfully!\n",
    "\n",
    "Models:\n",
    "  1. Logistic Regression    ✓\n",
    "  2. Random Forest          ✓\n",
    "  3. XGBoost                ✓\n",
    "  4. LSTM Neural Network    ✓\n",
    "\n",
    "Training Data:\n",
    "  - Training samples: {len(train_df):,}\n",
    "  - Validation samples: {len(val_df):,}\n",
    "  - Number of classes: {num_classes}\n",
    "  - Features (TF-IDF): {X_train.shape[1]:,}\n",
    "\n",
    "Best Model (Validation F1):\n",
    "  - {best_model_name}\n",
    "  - F1-Score: {best_f1:.4f}\n",
    "\n",
    "Saved Files:\n",
    "  ✓ logistic_model.pkl\n",
    "  ✓ random_forest_model.pkl\n",
    "  ✓ xgboost_model.pkl\n",
    "  ✓ lstm_model.h5\n",
    "  ✓ lstm_tokenizer.pkl\n",
    "  ✓ vectorizer.pkl\n",
    "  ✓ label_encoder.pkl\n",
    "  ✓ validation_results.csv\n",
    "\n",
    "Next Step: Run 04_evaluation.ipynb for detailed model comparison!\n",
    "\"\"\"\n",
    "\n",
    "print(summary)\n",
    "print(\"=\"*80)"
   ]
  }
 ],
 "metadata": {
  "kernelspec": {
   "display_name": "Python 3",
   "language": "python",
   "name": "python3"
  },
  "language_info": {
   "codemirror_mode": {
    "name": "ipython",
    "version": 3
   },
   "file_extension": ".py",
   "mimetype": "text/x-python",
   "name": "python",
   "nbconvert_exporter": "python",
   "pygments_lexer": "ipython3",
   "version": "3.8.0"
  }
 },
 "nbformat": 4,
 "nbformat_minor": 4
}